# 03 · Train the cloak generator

**Every code cell is self-contained.** On a fresh or reconnected runtime you
can run any one of them directly; you never need to rerun the cells above it.

- **Smoke run** (~20–30 min): 300 steps. Checks the loop works, and measures
  the real speed so it can project how long the main run will take.
- **Progress** (no GPU needed): status, ETA, loss and validation curves and the
  latest samples. Run it whenever.
- **Main run**: checkpoints every 500 steps and stops itself before the
  session limit (5.5 h on Colab; on Kaggle, 30 min before the 12 h hard stop, however long setup took). To continue, rerun only
  this cell: it resumes from `last.pt` and loses at most the steps since the
  last checkpoint.

What it saves in `ROOT/runs/<RUN_NAME>/`: `status.json` (step, GPU-minutes,
ETA, latest validation), `log.csv` (training curves), `val.csv` (protection
metrics every 500 steps), `samples/`, `generator_ema.pt` (the model to use)
and `last.pt` (full resume state).

Each person and each experiment needs its own `RUN_NAME`. Two runs writing to
one folder would resume from each other's checkpoints.

**Kaggle:** attach the 01_data output (Add Input → Your Work). Train with
**Save Version → Save & Run All**, so it runs in the background and its output
(the run folder) is kept; set `RUN_SMOKE = False` once the smoke run looks
right, or every version repeats it. To continue in the next session, attach
the previous version's output as an input too (if Kaggle won't attach a
notebook's own output, use Output → New Dataset and attach that): the cells
find `runs/<RUN_NAME>/last.pt` there and resume from it.

**Colab:** results go to Google Drive (`MyDrive/soteria`). Runtime → T4 GPU.

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a
phone-verified account). Results go to `/kaggle/working/soteria` and are only
kept when you save a version with **Save Version → Save & Run All**; from an
interactive session, download what you need from the Output panel before it ends.
Secrets (`HF_TOKEN`, `GITHUB_TOKEN` for a private fork) go in Add-ons → Secrets.

In [ ]:
# ---- setup: lets this cell run on its own, on Colab or Kaggle; safe to rerun ----
REPO_URL = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH   = "main"
import importlib, os, subprocess, sys
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/input")
if not KAGGLE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")                # Colab: results live on Drive
CODE = "/tmp/Project-Soteria" if KAGGLE else "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    _token = _secret("GITHUB_TOKEN")             # only needed for a private fork
    _git("clone", "-q", "-b", BRANCH, REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.runtime as rt; importlib.reload(rt)
PLATFORM = rt.platform()
ROOT, DATA = rt.prepare(data=True, gpu=True, requirements=('requirements.txt',))
# ---------------------------------------------------------------------------------

RUN_SMOKE   = True         # set False once checked (Kaggle "Save & Run All" runs every cell)
TRAIN_HOURS = 12           # GPU-hours you can give the main run in total; prints the STEPS that fit
if RUN_SMOKE:
    import glob, pandas as pd, yaml
    from IPython.display import Image as Show, display
    get_ipython().system(f"python -m soteria.train --config configs/smoke.yaml out_dir={ROOT}/runs/smoke data_root={DATA}")
    log = pd.read_csv(f"{ROOT}/runs/smoke/log.csv")
    img_per_s = log["img_per_s"].iloc[len(log) // 2:].median()     # skip warm-up
    main = yaml.safe_load(open("configs/base_256.yaml"))
    hours = main["steps"] * main["batch_size"] / img_per_s / 3600
    session_h = rt.SESSION_MIN[PLATFORM] / 60
    print(f"{img_per_s:.1f} img/s -> main run ({main['steps']} steps x {main['batch_size']}) ~ {hours:.1f} GPU-hours "
          f"~ {hours / session_h:.1f} sessions of {session_h:.1f} h on {PLATFORM}")
    fit = int(TRAIN_HOURS * 3600 * img_per_s / main["batch_size"] // 500 * 500)
    print(f"{TRAIN_HOURS} GPU-hours of training fit ~{fit} steps -> set STEPS = {fit} in the main-run cell")
    samples = sorted(glob.glob(f"{ROOT}/runs/smoke/samples/*.png"))
    if samples:
        display(Show(samples[-1]))

In [ ]:
# ---- setup: lets this cell run on its own, on Colab or Kaggle; safe to rerun ----
REPO_URL = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH   = "main"
import importlib, os, subprocess, sys
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/input")
if not KAGGLE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")                # Colab: results live on Drive
CODE = "/tmp/Project-Soteria" if KAGGLE else "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    _token = _secret("GITHUB_TOKEN")             # only needed for a private fork
    _git("clone", "-q", "-b", BRANCH, REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.runtime as rt; importlib.reload(rt)
PLATFORM = rt.platform()
ROOT, DATA = rt.prepare(data=False, gpu=False, requirements=())
# ---------------------------------------------------------------------------------

# ---- this run ----------------------------------------------------------------
RUN_NAME  = "base_256"     # one folder per experiment AND per person, e.g. "ahnaf_base_256"
CONFIG    = "configs/base_256.yaml"
OVERRIDES = ""             # e.g. "clip_floor=0.3 face_floor=-0.1 w_vae=4". Must not change
                           # while resuming one RUN_NAME: train.py refuses a mismatch.
OUT = ROOT / "runs" / RUN_NAME
rt.restore_run(OUT)        # Kaggle: continue from an earlier saved version attached as input

import glob, json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image as Show, display
if not (OUT / "status.json").exists():
    print(f"no run in {OUT} yet")
else:
    status = json.loads((OUT / "status.json").read_text())
    print(json.dumps({k: v for k, v in status.items() if k != "last_val"}, indent=1))
    log = pd.read_csv(OUT / "log.csv")
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
    log.plot(x="step", y=[c for c in log if c.startswith("loss_")], ax=axes[0], title="train loss terms")
    log.plot(x="step", y=[c for c in ("lpips_raw", "clip_cos", "face_cos") if c in log], ax=axes[1], title="train")
    if (OUT / "val.csv").exists():
        val = pd.read_csv(OUT / "val.csv")
        val.plot(x="step", y=[c for c in val if c.startswith("jpeg75/") and not c.endswith("_shift")],
                 ax=axes[2], title="val after JPEG q75 (lower = better)")
        display(val.tail(3).set_index("step").T.round(4))
    plt.tight_layout(); plt.show()
    samples = sorted(glob.glob(f"{OUT}/samples/*.png"))
    if samples:
        print(samples[-1], "(rows: clean / cloaked / |delta| x10)")
        display(Show(samples[-1]))

In [ ]:
# ---- setup: lets this cell run on its own, on Colab or Kaggle; safe to rerun ----
REPO_URL = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH   = "main"
import importlib, os, subprocess, sys
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/input")
if not KAGGLE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")                # Colab: results live on Drive
CODE = "/tmp/Project-Soteria" if KAGGLE else "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    _token = _secret("GITHUB_TOKEN")             # only needed for a private fork
    _git("clone", "-q", "-b", BRANCH, REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.runtime as rt; importlib.reload(rt)
PLATFORM = rt.platform()
ROOT, DATA = rt.prepare(data=True, gpu=True, requirements=('requirements.txt',))
# ---------------------------------------------------------------------------------

# ---- this run ----------------------------------------------------------------
RUN_NAME  = "base_256"     # one folder per experiment AND per person, e.g. "ahnaf_base_256"
CONFIG    = "configs/base_256.yaml"
OVERRIDES = ""             # e.g. "clip_floor=0.3 face_floor=-0.1 w_vae=4". Must not change
                           # while resuming one RUN_NAME: train.py refuses a mismatch.
OUT = ROOT / "runs" / RUN_NAME
rt.restore_run(OUT)        # Kaggle: continue from an earlier saved version attached as input

RUN_MAIN   = False         # set True to train (Kaggle "Save & Run All" runs every cell)
STEPS      = 20000         # total steps for the whole run; size it from the smoke run's suggestion.
                           # Set it before the first session: the LR schedule is built from it.
BUDGET_MIN = rt.session_budget_min()   # training minutes in THIS session. Kaggle: all that is left
                           # of the 12 h session limit minus 30 min, so the run stops itself and is saved
                           # rather than being killed. Colab: 330. Or set a number.
if RUN_MAIN:
    print(f"training for up to {BUDGET_MIN} min this session, then saving; rerun to resume")
    get_ipython().system(f"python -m soteria.train --config {CONFIG} out_dir={OUT} data_root={DATA} "
                         f"steps={STEPS} budget_min={BUDGET_MIN} {OVERRIDES}")
    print((OUT / "status.json").read_text() if (OUT / "status.json").exists() else "no status written")
else:
    print("RUN_MAIN is False: main run skipped")